In [9]:
import os
import numpy as np
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf

In [10]:
IMG_SIZE = 64

def get_label(folder_name):
    if folder_name.lower() == "healthy":
        return 0
    else:
        return 1

In [11]:
def load_dataset(base_path):
    data = []
    labels = []

    for class_name in os.listdir(base_path):
        class_path = os.path.join(base_path, class_name)

        if not os.path.isdir(class_path):
            continue

        label = get_label(class_name)

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            img = cv2.imread(img_path)
            if img is None:
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0   # normalize

            data.append(img)
            labels.append(label)

    return np.array(data), np.array(labels)

In [12]:
X_train, y_train = load_dataset("Training")
X_val, y_val = load_dataset("Validation")
X_test, y_test = load_dataset("Testing")

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (80, 64, 64, 3)
Validation shape: (40, 64, 64, 3)
Test shape: (20, 64, 64, 3)


In [13]:
y_train = to_categorical(y_train, 2)
y_val = to_categorical(y_val, 2)
y_test_cat = to_categorical(y_test, 2)

In [14]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(2, activation='softmax'))

c:\Users\admin\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [17]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_val, y_val),
    batch_size=16
)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 12s 474ms/step - accuracy: 0.7125 - loss: 0.5862 - val_accuracy: 0.8000 - val_loss: 0.4878
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 189ms/step - accuracy: 0.8000 - loss: 0.4965 - val_accuracy: 0.8000 - val_loss: 0.4417
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step - accuracy: 0.8000 - loss: 0.4489 - val_accuracy: 0.8000 - val_loss: 0.3948
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step - accuracy: 0.8000 - loss: 0.4265 - val_accuracy: 0.8000 - val_loss: 0.3159
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 210ms/step - accuracy: 0.8000 - loss: 0.3834 - val_accuracy: 0.8000 - val_loss: 0.2853
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 163ms/step - accuracy: 0.8000 - loss: 0.3692 - val_accuracy: 0.8250 - val_loss: 0.2720
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 271ms/step - accuracy: 0.8125 - loss: 0.3948 - val_accuracy: 0.8000 - val_loss: 0.2704
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 385ms/step - accuracy: 0.8000 - loss: 0.3634 - val_accuracy: 0.8000 - val_loss

In [18]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Test Accuracy:", accuracy_score(y_test, y_pred_classes))
print("\nClassification Report:\n", classification_report(y_test, y_pred_classes))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 364ms/step
Test Accuracy: 0.85

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.25      0.40         4
           1       0.84      1.00      0.91        16

    accuracy                           0.85        20
   macro avg       0.92      0.62      0.66        20
weighted avg       0.87      0.85      0.81        20



In [19]:
model.save("leaf_model.h5")